In [1]:
# LOADING THE DATA
import pandas as pd
import json

ledger = pd.read_csv("/content/ledger.csv")
gateway = pd.read_csv("/content/gateway.csv")

print("--- Q1: Load both files ---")
print(f"Ledger records loaded: {len(ledger)}")
print(f"Gateway records loaded: {len(gateway)}")

--- Q1: Load both files ---
Ledger records loaded: 10
Gateway records loaded: 9


In [2]:
#Check duplicates and nulls

print("Ledger Nulls:\n", ledger.isnull().sum())
print("Gateway Nulls:\n", gateway.isnull().sum())

print(f"\nLedger Duplicates: {ledger.duplicated().sum()}")
print(f"\nGateway Duplicates: {gateway.duplicated().sum()}")

Ledger Nulls:
 transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64
Gateway Nulls:
 transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64

Ledger Duplicates: 0

Gateway Duplicates: 0


In [3]:
# 3 Create a list of IDs that exist in the Gateway
gateway_id_list = gateway['transaction_id'].tolist()

missing_in_gateway = ledger[ledger['transaction_id'].isin(gateway_id_list) == False].copy()
missing_in_gateway['issue_type'] = 'Missing in Gateway'

print("\n--- Q3: Identify records missing in gateway ---")
print(f"Count: {len(missing_in_gateway)}")
print(missing_in_gateway[['transaction_id', 'amount_usd' , 'issue_type']])

missing_in_gateway.to_csv('missing_in_gateway.csv', index=False)


--- Q3: Identify records missing in gateway ---
Count: 2
  transaction_id  amount_usd          issue_type
3           R004      2100.0  Missing in Gateway
9           R010      2500.0  Missing in Gateway


In [4]:
# 4 Create a list of IDs that exist in the Ledger
ledger_id_list = ledger['transaction_id'].tolist()

missing_in_ledger = gateway[gateway['transaction_id'].isin(ledger_id_list) == False].copy()
missing_in_ledger['issue_type'] = 'Missing in Ledger'

print("\n--- Q4: Identify records missing in ledger ---")
print(f"Count: {len(missing_in_ledger)}")
print(missing_in_ledger[['transaction_id', 'amount_usd', 'issue_type']])
missing_in_ledger.to_csv('missing_in_ledger.csv', index=False)


--- Q4: Identify records missing in ledger ---
Count: 1
  transaction_id  amount_usd         issue_type
8           R011      1800.0  Missing in Ledger


In [5]:
# 5  Filter for records that appear in both files
ledger_common = ledger[ledger['transaction_id'].isin(gateway['transaction_id'])]
gateway_common = gateway[gateway['transaction_id'].isin(ledger['transaction_id'])]

# Merge specific columns to compare amounts
merged_amounts = pd.merge(
    ledger_common[['transaction_id', 'amount_usd']],
    gateway_common[['transaction_id', 'amount_usd']],
    on='transaction_id',
    suffixes=('_ledger', '_gateway')
)

merged_amounts['amount_difference'] = merged_amounts['amount_usd_ledger'] - merged_amounts['amount_usd_gateway']
amount_mismatch = merged_amounts[merged_amounts['amount_difference'] != 0].copy()
amount_mismatch['issue_type'] = 'Amount Mismatch'

print("\n--- Q5: Identify amount mismatches ---")
print(f"Count: {len(amount_mismatch)}")
print(amount_mismatch)

amount_mismatch.to_csv('amount_mismatches.csv', index=False)


--- Q5: Identify amount mismatches ---
Count: 2
  transaction_id  amount_usd_ledger  amount_usd_gateway  amount_difference  \
1           R002              850.0               900.0              -50.0   
6           R008              640.0               600.0               40.0   

        issue_type  
1  Amount Mismatch  
6  Amount Mismatch  


In [6]:
# 6  Merge specific columns to compare status
merged_status = pd.merge(
    ledger_common[['transaction_id', 'status']],
    gateway_common[['transaction_id', 'status']],
    on='transaction_id',
    suffixes=('_ledger', '_gateway')
)

status_mismatch = merged_status[merged_status['status_ledger'] != merged_status['status_gateway']].copy()
status_mismatch['issue_type'] = 'Status Mismatch'

print("\n--- Q6: Identify status mismatches ---")
print(f"Count: {len(status_mismatch)}")
print(status_mismatch)

status_mismatch.to_csv('status_mismatches.csv', index=False)


--- Q6: Identify status mismatches ---
Count: 1
  transaction_id status_ledger status_gateway       issue_type
3           R005       success         failed  Status Mismatch


In [7]:
# Combine all identified issues into one dataframe
reconciliation_report = pd.concat([
    missing_in_gateway,
    missing_in_ledger,
    amount_mismatch,
    status_mismatch
], ignore_index=True)

reconciliation_report.to_csv('reconciliation_report.csv', index=False)
print("\n--- Q7: Build a final reconciliation report ---")
print("Report saved as 'reconciliation_report.csv'")
print(f"Total entries in report: {len(reconciliation_report)}")

reconciliation_report.to_csv('reconciliation_report.csv', index=False)


--- Q7: Build a final reconciliation report ---
Report saved as 'reconciliation_report.csv'
Total entries in report: 6


In [8]:
summary_metrics = {
    "total_ledger_rows": len(ledger),
    "total_gateway_rows": len(gateway),
    "missing_in_gateway_count": len(missing_in_gateway),
    "missing_in_ledger_count": len(missing_in_ledger),
    "amount_mismatch_count": len(amount_mismatch),
    "status_mismatch_count": len(status_mismatch),
    "reconciliation_issue_count": len(reconciliation_report),
    "ledger_total_amount": float(ledger['amount_usd'].sum()),
    "gateway_total_amount": float(gateway['amount_usd'].sum()),
    "amount_at_risk": float(reconciliation_report['amount_usd'].sum() if 'amount_usd' in reconciliation_report.columns else 0)
}

# 2. Convert to JSON string with neat indentation
json_text = json.dumps(summary_metrics, indent=4)

# 3. Save the file (Simple Method)
f = open('summary_metrics.json', 'w')
f.write(json_text)
f.close()

print("--- Summary metrics generated ---")
print(json_text)

--- Summary metrics generated ---
{
    "total_ledger_rows": 10,
    "total_gateway_rows": 9,
    "missing_in_gateway_count": 2,
    "missing_in_ledger_count": 1,
    "amount_mismatch_count": 2,
    "status_mismatch_count": 1,
    "reconciliation_issue_count": 6,
    "ledger_total_amount": 23340.0,
    "gateway_total_amount": 20550.0,
    "amount_at_risk": 6400.0
}


In [9]:
#Part 4: JSON Normalization

In [10]:
import pandas as pd
import json

with open('api_response_sample.json','r') as file:
  api_data = json.load(file)

api_data

{'generated_at': '2026-03-07T10:00:00Z',
 'source': 'QuickPay Settlement API',
 'batches': [{'batch_id': 'B001',
   'merchant': {'merchant_id': 'M001',
    'merchant_name': 'Alpha Mart',
    'region': 'APAC'},
   'settlements': [{'settlement_id': 'S001',
     'amount_usd': 1520.5,
     'status': 'settled',
     'processed_at': '2026-03-07T08:10:00Z',
     'bank': {'name': 'Bank A', 'country': 'IN'}},
    {'settlement_id': 'S002',
     'amount_usd': 980.0,
     'status': 'pending',
     'processed_at': '2026-03-07T08:45:00Z',
     'bank': {'name': 'Bank A', 'country': 'IN'}},
    {'settlement_id': 'S003',
     'amount_usd': 640.0,
     'status': 'settled',
     'processed_at': '2026-03-07T09:15:00Z',
     'bank': {'name': 'Bank B', 'country': 'SG'}}]},
  {'batch_id': 'B002',
   'merchant': {'merchant_id': 'M004',
    'merchant_name': 'Delta Travels',
    'region': 'US'},
   'settlements': [{'settlement_id': 'S004',
     'amount_usd': 2100.0,
     'status': 'settled',
     'processed_at'

In [11]:
all_settlements = []

for merchant_batch in api_data['batches']:
  current_batch_id = merchant_batch['batch_id']
  current_merchant_id = merchant_batch['merchant']['merchant_id']
  current_merchant_name = merchant_batch['merchant']['merchant_name']
  current_region = merchant_batch['merchant']['region']

  for settlement in merchant_batch['settlements']:
    current_settlement = {
        'batch_id': current_batch_id,
        'merchant_id': current_merchant_id,
        'region' : current_region,
        'settlement_id' : settlement['settlement_id'],
        'amount_usd' : settlement['amount_usd'],
        'status' : settlement['status'],
        'processed_at' : settlement['processed_at'],
        'bank_name' : settlement['bank']['name'],
        'bank_country' : settlement['bank']['country'],
    }
    all_settlements.append(current_settlement)

print(all_settlements[:2])
print(f'Total normalized settlements: {len(all_settlements)}')


[{'batch_id': 'B001', 'merchant_id': 'M001', 'region': 'APAC', 'settlement_id': 'S001', 'amount_usd': 1520.5, 'status': 'settled', 'processed_at': '2026-03-07T08:10:00Z', 'bank_name': 'Bank A', 'bank_country': 'IN'}, {'batch_id': 'B001', 'merchant_id': 'M001', 'region': 'APAC', 'settlement_id': 'S002', 'amount_usd': 980.0, 'status': 'pending', 'processed_at': '2026-03-07T08:45:00Z', 'bank_name': 'Bank A', 'bank_country': 'IN'}]
Total normalized settlements: 6


In [12]:
# 4 Convert the list into the table
df = pd.DataFrame(all_settlements)

# 5  Clean column names to be lowercase

df.columns = [column.lower() for column in df.columns]

# 6 Convert the date field to a proper format

df['processed_at'] = pd.to_datetime(df['processed_at'])

In [13]:
# 7 Save the output to the requested folder

output_path = 'api_normalized.csv'
df.to_csv(output_path, index=False)

print("Success! Your JSON data is now a flat table.")
print(df.head())

Success! Your JSON data is now a flat table.
  batch_id merchant_id region settlement_id  amount_usd   status  \
0     B001        M001   APAC          S001      1520.5  settled   
1     B001        M001   APAC          S002       980.0  pending   
2     B001        M001   APAC          S003       640.0  settled   
3     B002        M004     US          S004      2100.0  settled   
4     B002        M004     US          S005       500.0   failed   

               processed_at bank_name bank_country  
0 2026-03-07 08:10:00+00:00    Bank A           IN  
1 2026-03-07 08:45:00+00:00    Bank A           IN  
2 2026-03-07 09:15:00+00:00    Bank B           SG  
3 2026-03-07 08:20:00+00:00    Bank C           US  
4 2026-03-07 08:50:00+00:00    Bank C           US  
